# Ferroelectric Switching of HfO₂ along the S:T Pathway

## 0. Introduction

This notebook builds the initial and final structures of the S:T polarization reversal of orthorhombic Pca2₁ HfO₂ from the manuscript:

> **Xinpeng Mu, Yao Wu, Binjian Zeng, Jie Jiang, Yichun Zhou, Lu Yin, Min Liao, and Qiong Yang**
> "Polarization switching of HfO2 ferroelectric in bulk and electrode/ferroelectric/electrode heterostructure"
> npj Computational Materials 11, 126 (2025).
> [DOI: 10.1038/s41524-025-01633-2](https://doi.org/10.1038/s41524-025-01633-2)

The manuscript names the pathways (p. 2): "the three-fold coordinated O shifting **through** (denoted by T) or **not** through (denoted by N) Hf-atomic planes along the polarization (z-) direction leads to opposite polarization reversals. And as seen along the x-direction, the polarization reversal may be realized by the **straight** z-direction shifting of three-fold coordinated O (denoted by S) [...]".

The initial structure is Fig. 1(a); the S:T pathway takes it to Fig. 1(c), both seen along x. In the final structure each three-fold coordinated O has moved straight along z through the Hf plane below it, to the same distance on the other side. The two structures are saved by name for the [simulation notebook](perturbation_ferroelectric_switching_hafnium_dioxide_SIMULATION.ipynb).

<img src="https://github.com/Exabyte-io/documentation/raw/5afed9f2eadf01730bb9bb2c89a87d1ca2ecdeb2/images/tutorials/materials/defects/perturbation_ferroelectric_switching_hafnium_dioxide/0-figure-from-manuscript.webp" alt="Polarization reversal pathways of Pca2₁ HfO₂" width="400"/>

## 1. Prepare the Environment
### 1.1. Set up the parameters

In [ ]:
# Pca2₁ HfO₂ in the orientation of Fig. 1: a along x, the polarization along z
MATERIAL_NAME = "HfO2, Hafnium IV Oxide, ORC (Pca2_1) 3D (Bulk), mp-685097"

# The atoms that move: O with three Hf neighbors (three-fold coordinated)
MOVING_ELEMENT = "O"
MOVING_COORDINATION = 3
NEIGHBOR_ELEMENT = "Hf"
NEIGHBOR_RADIUS = 2.5  # in Angstroms; the O-Hf bonds are 2.02-2.23 Å, the next Hf is 3.25 Å away

INITIAL_MATERIAL_NAME = "HfO2 Pca21"
FINAL_MATERIAL_NAME = "HfO2 Pca21 S:T switched"

### 1.2. Install Packages
The step executes only in Pyodide environment. For other environments, the packages should be installed via `pip install` (see [README](../../README.ipynb)).

In [ ]:
from mat3ra.notebooks_utils.packages import install_packages

await install_packages("made|specific_examples")

### 1.3. Get input material

In [ ]:
from mat3ra.made.material import Material
from mat3ra.standata.materials import Materials

material = Material.create(Materials.get_by_name_first_match(MATERIAL_NAME))

## 2. Create the Switched Structure
### 2.1. Find the three-fold coordinated O
The O atoms with `MOVING_COORDINATION` Hf atoms within `NEIGHBOR_RADIUS`, periodic images included.

In [ ]:
from mat3ra.made.tools.analyze.other import get_atom_indices_within_radius_pbc

elements = material.basis.elements.values
moving_atom_ids = []
for atom_id, element in enumerate(elements):
    neighbor_ids = get_atom_indices_within_radius_pbc(material, atom_index=atom_id, radius=NEIGHBOR_RADIUS)
    neighbor_elements = [elements[neighbor_id] for neighbor_id in neighbor_ids]
    if element == MOVING_ELEMENT and neighbor_elements.count(NEIGHBOR_ELEMENT) == MOVING_COORDINATION:
        moving_atom_ids.append(atom_id)

### 2.2. Shift them through the Hf plane
In Pca2₁ all four three-fold coordinated O sit at the same height above the Hf plane below them, so one shift, straight along z by twice that height, moves each through its plane.

In [ ]:
from mat3ra.made.tools.operations.core.unary import translate_atoms

cartesian_material = material.clone()
cartesian_material.to_cartesian()
coordinates = cartesian_material.basis.coordinates.values
moving_z = coordinates[moving_atom_ids[0]][2]
neighbor_z_values = [coordinate[2] for element, coordinate in zip(elements, coordinates) if element == NEIGHBOR_ELEMENT]
neighbor_plane_z = max(z for z in neighbor_z_values if z < moving_z)
switched_material = translate_atoms(material, moving_atom_ids, [0, 0, 2 * (neighbor_plane_z - moving_z)])

## 3. Visualize the Structures

In [ ]:
from mat3ra.notebooks_utils.ipython.entity.material.visualize import visualize_materials as visualize

material.name = INITIAL_MATERIAL_NAME
switched_material.name = FINAL_MATERIAL_NAME
materials = [material, switched_material]
visualize([{"material": saved_material, "title": saved_material.name} for saved_material in materials], viewer="wave")

## 4. Pass data to the outside runtime

In [ ]:
from mat3ra.notebooks_utils.io import download_content_to_file
from mat3ra.notebooks_utils.material import set_materials

set_materials(materials)
for saved_material in materials:
    download_content_to_file(saved_material.to_json(), f"{saved_material.name}.json")

## References

[1] X. Mu, Y. Wu, B. Zeng, J. Jiang, Y. Zhou, L. Yin, M. Liao, and Q. Yang, "Polarization switching of HfO2 ferroelectric in bulk and electrode/ferroelectric/electrode heterostructure", npj Comput. Mater. 11, 126 (2025). [DOI: 10.1038/s41524-025-01633-2](https://doi.org/10.1038/s41524-025-01633-2)